<a href="https://colab.research.google.com/github/dougyd92/ML-Foudations/blob/main/Projects/Project_4_Handwriting_Neural_Net__keras_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Neural Network Classification of Handwritten Letters

## Overview

In this project, you will design, implement, and train a neural network to classify handwritten letters using the EMNIST Letters dataset. This dataset contains 28×28 grayscale images of handwritten letters (A–Z), providing a more challenging classification task than the classic MNIST digit dataset.

Your classifier will be evaluated on a separate, held-out test set that you will not have access to during development. This test set consists of handwritten letters created independently from the EMNIST dataset, so your model must generalize well to truly unseen data.

## Learning Objectives

By completing this project, you will:

- Gain hands-on experience loading and preprocessing image data for neural network training
- Design and implement neural network architectures using TensorFlow/Keras
- Apply techniques such as data augmentation, regularization, and hyperparameter tuning
- Evaluate model performance and iterate on your design
- Develop intuition for the tradeoffs between model complexity, training time, and generalization

## The Competition

This project is a competition. The student whose model achieves the **highest accuracy on the held-out test set wins** and will receive a shoutout on LinkedIn.

May the best model win!

## Submission Requirements

1. This completed Jupyter notebook with all code cells executed
2. Your exported model saved as `submission_model.keras` (generated by the submission cell at the end)

That's it: **two files.** The submission cell handles model export automatically. No separate prediction script needed.

## Important Notes

- **EMNIST Orientation**: EMNIST images are stored transposed and flipped. You must correct this when loading the data.
- **Label Indexing**: EMNIST Letters uses 1-indexed labels (1–26 for A–Z). You will need to convert to 0-indexed (0–25) for compatibility with Keras's `sparse_categorical_crossentropy`.
- **No Pretrained Models**: You must train your model from scratch. Do not use pretrained weights or transfer learning.
- **Parameter Limit**: Your model must have **1 million parameters or fewer**. Models exceeding this limit will be disqualified. Code is provided below to check your parameter count.
- **Reproducibility**: Set random seeds for reproducibility and document any non-deterministic choices.
- **Hardware**: If running in Colab, be sure to go to "Change runtime type" and select either a GPU or TPU instead of the default CPU. If running on your own machine, be sure you have the correct drivers to take advantage of your GPU.

---

## Section 1: Setup and Imports

Import all necessary libraries for the project. You will need at minimum:

- TensorFlow and Keras for model building and training
- TensorFlow Datasets (`tensorflow_datasets`) for data loading
- NumPy for numerical operations
- Matplotlib for visualization

You may also find these useful:
- `tf.keras.layers` for building your model architecture
- `tf.keras.callbacks` for early stopping, checkpointing, and learning rate scheduling
- `tf.keras.preprocessing.image` or `tf.image` for data augmentation
- `sklearn.metrics` for additional evaluation metrics

Set your random seeds here for reproducibility.

In [ ]:
# Your imports here


In [ ]:
# Set random seeds for reproducibility
# Consider: tf.random.set_seed(), numpy random seed


In [ ]:
# Check for GPU availability
# tf.config.list_physical_devices('GPU')


---

## Section 2: Data Loading and Exploration

Load the EMNIST Letters dataset using `tensorflow_datasets`. The dataset will be automatically downloaded if not present.

### Tasks:

1. Load the training and test splits of EMNIST Letters
2. Explore the dataset: How many samples? What are the image dimensions? How many classes?
3. Visualize several example images from different classes
4. Check the class distribution—is the dataset balanced?

### Important: Fixing EMNIST Orientation

EMNIST images are stored transposed and flipped compared to how they should appear. You need to apply a transformation to correct this:

```python
# The image needs to be rotated and flipped to appear correctly
# This can be done with tf.image.rot90 and tf.image.flip_left_right
```

Verify your orientation fix by visualizing some images—letters should be readable.

In [ ]:
# Load EMNIST Letters dataset
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np

def fix_emnist_orientation(image, label):
    """EMNIST images are stored transposed and flipped. This corrects them."""
    image = tf.image.rot90(image, k=3)      # Rotate -90 degrees
    image = tf.image.flip_left_right(image)  # Flip horizontally
    return image, label

# Load training and test sets
(ds_train_raw, ds_test_raw), ds_info = tfds.load(
    'emnist/letters',
    split=['train', 'test'],
    with_info=True,
    as_supervised=True
)

# Apply orientation fix and convert to numpy for exploration
# Images are uint8 [0, 255], shape (28, 28, 1)
train_images, train_labels = [], []
for img, lbl in ds_train_raw.map(fix_emnist_orientation):
    train_images.append(img.numpy())
    train_labels.append(lbl.numpy())
train_images = np.array(train_images)
train_labels = np.array(train_labels)

test_images, test_labels = [], []
for img, lbl in ds_test_raw.map(fix_emnist_orientation):
    test_images.append(img.numpy())
    test_labels.append(lbl.numpy())
test_images = np.array(test_images)
test_labels = np.array(test_labels)

# Note: EMNIST Letters labels are 1-indexed (1-26 for A-Z)
# You will need to subtract 1 to make them 0-indexed for Keras
print(f"Training samples: {len(train_images)}")
print(f"Test samples: {len(test_images)}")
print(f"Image shape: {train_images[0].shape}")
print(f"Pixel range: {train_images.min()} to {train_images.max()}")
print(f"Label range: {train_labels.min()} to {train_labels.max()}")

In [ ]:
# The dataset is already loaded above. Add any additional exploration here.
# For example, you might want to check unique classes, compute statistics, etc.


In [ ]:
# Visualize sample images
import matplotlib.pyplot as plt

def label_to_letter(label):
    """Convert 1-indexed label to letter. Label 1 = 'A', 2 = 'B', etc."""
    return chr(label - 1 + ord('A'))

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_images[i].squeeze(), cmap='gray')
    ax.set_title(f'{label_to_letter(train_labels[i])}')
    ax.axis('off')
plt.suptitle('Sample EMNIST Letters (orientation corrected)')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze class distribution
# Are all letters equally represented in the training set?


### Notes

Use this space to jot down observations about the dataset—anything that might inform your modeling decisions. For example: Are some letters likely to be confused with each other? Is the dataset balanced?

*Your notes:*



---

## Section 3: Data Preprocessing and Augmentation

Prepare your data for training. This section is critical for achieving good generalization.

### Required Preprocessing:

- Fix the EMNIST transpose/flip issue (already done in Section 2)
- Scale pixel values from [0, 255] to [0, 1] (divide by 255.0)
- Normalize pixel values (consider computing dataset mean/std, or use [0, 1] range directly)
- Adjust labels from 1-indexed to 0-indexed (subtract 1)

### Suggested Data Augmentation (choose based on your experimentation):

- **Random rotation**: Small rotations (±10-15°) simulate natural writing variation
- **Random translation/zoom**: `tf.keras.layers.RandomTranslation`, `RandomZoom`
- **Random contrast**: `tf.keras.layers.RandomContrast`
- **Gaussian noise**: Adds robustness to noise in input

Augmentation can be applied via Keras preprocessing layers (included in the model) or via `tf.image` ops in a `tf.data` pipeline.

**Important**: Data augmentation should typically only be applied to training data, not validation or test data.

### Creating a Validation Split:

You should create a validation set from the training data to monitor for overfitting and tune hyperparameters. A typical split is 80-90% training, 10-20% validation.

### ⚠️ Track Your Normalization Values

If you normalize your images beyond simple [0, 1] scaling (e.g., zero-centering with dataset mean/std), **write down the exact mean and std values you used.** You will need them in the submission cell at the end of the notebook. The submission cell bakes your normalization into the exported model so that grading works consistently across all submissions.

If you only scale to [0, 1] (divide by 255) with no further normalization, you don't need to do anything extra.

In [ ]:
# Preprocess images and labels
# Scale pixel values to [0, 1] and convert labels to 0-indexed


In [ ]:
# Create train/validation split
# Hint: Use sklearn.model_selection.train_test_split or manual index slicing


In [ ]:
# Define data augmentation for training data
# Options: Keras preprocessing layers, tf.image ops in a tf.data pipeline, or
#          ImageDataGenerator (older API but still works)


In [ ]:
# Create tf.data.Dataset pipelines for train, validation, and test
# Include batching, shuffling (for train), and prefetching
# Consider appropriate batch sizes (32, 64, 128 are common choices)
# Example:
#   train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
#   train_ds = train_ds.shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)


In [ ]:
# Visualize some augmented training samples to verify your transforms look reasonable


### Preprocessing Notes

Document any preprocessing decisions worth remembering—normalization strategy, augmentation choices, batch size, etc.

*Your notes:*



---

## Section 4: Model Architecture

Design and implement your neural network architecture. You have flexibility in your design choices, but your model must be trained from scratch (no pretrained weights).


### ⚠️ Parameter Limit: 1,000,000

Your model must have **1 million trainable parameters or fewer**. Models exceeding this limit will be disqualified. Use the parameter counting code provided below to verify your model is within the limit.

### Architecture Options to Consider:

**Fully Connected Networks:**
- Simpler to implement
- May require more parameters to achieve good performance
- Don't inherently capture spatial structure

**Convolutional Neural Networks (CNNs):**
- Better suited for image data
- Capture local spatial patterns
- Typically more parameter-efficient for image tasks

### Design Considerations:

- **Depth**: How many layers? Deeper networks can learn more complex features but are harder to train
- **Width**: How many neurons/filters per layer?
- **Activation functions**: ReLU is standard, but consider LeakyReLU, ELU, or GELU
- **Regularization**: Dropout, batch normalization, weight decay
- **Pooling**: Max pooling, average pooling, or strided convolutions

### Suggested Starting Points:

For a CNN, a reasonable starting architecture might be:
- 2-3 convolutional blocks (Conv2D → activation → pooling)
- 1-2 Dense layers
- Dropout for regularization
- Output layer with 26 neurons (one per letter)

Start simple and add complexity as needed!

In [ ]:
# Define your neural network architecture
# You can use Sequential or Functional API

# Example skeleton using Sequential:
# model = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(28, 28, 1)),
#     # Add your layers here
# ])

# Example skeleton using Functional API:
# inputs = tf.keras.Input(shape=(28, 28, 1))
# x = ...
# outputs = tf.keras.layers.Dense(26)(x)
# model = tf.keras.Model(inputs, outputs)


In [ ]:
# If you haven't already, build/instantiate your model
# Verify it accepts the correct input shape: (batch_size, 28, 28, 1)


In [ ]:
# Print model summary and check parameter limit
PARAMETER_LIMIT = 1_000_000

model.summary()

num_params = model.count_params()
print(f"\n" + "=" * 50)
print(f"Total trainable parameters: {num_params:,}")
print(f"Parameter limit:            {PARAMETER_LIMIT:,}")
print(f"=" * 50)

if num_params > PARAMETER_LIMIT:
    print(f"\n⚠️  WARNING: Your model has {num_params - PARAMETER_LIMIT:,} parameters over the limit!")
    print(f"    You must reduce your model size to be eligible for the competition.")
else:
    print(f"\n✓ Your model is within the parameter limit.")
    print(f"  Remaining budget: {PARAMETER_LIMIT - num_params:,} parameters")

In [ ]:
# Additional model inspection if needed
# For example, check layer output shapes or test with a dummy batch


In [ ]:
# Verify the model works by passing a sample batch through it


### Architecture Notes

Document your architectural choices—why this type of network, how you chose depth/width, regularization techniques, parameter count, etc.

*Your notes:*



---

## Section 5: Training Setup

Configure the training process using `model.compile()`.

### Loss Function:

For multi-class classification with integer labels, use `sparse_categorical_crossentropy`. If your model's output layer does not include a softmax activation, set `from_logits=True`.

### Optimizer Options:

- **SGD**: Classic choice, may require careful learning rate tuning and momentum
- **Adam**: Adaptive learning rates, often works well out of the box (`tf.keras.optimizers.Adam`)
- **AdamW**: Adam with decoupled weight decay (`tf.keras.optimizers.AdamW`)

### Callbacks (optional but recommended):

- **ModelCheckpoint**: Save the best model based on validation accuracy
- **EarlyStopping**: Stop training when validation loss stops improving
- **ReduceLROnPlateau**: Reduce learning rate when validation loss plateaus
- **CSVLogger**: Log training metrics to a file

### Hyperparameters to Tune:

- Learning rate (try values like 0.001, 0.01, 0.0001)
- Weight decay (L2 regularization via optimizer or kernel_regularizer)
- Number of epochs
- Optimizer-specific parameters (momentum, beta values for Adam)

In [ ]:
# Compile your model
# Choose loss function, optimizer, and metrics to track
# Example: model.compile(optimizer=..., loss=..., metrics=['accuracy'])


In [ ]:
# Define callbacks
# Consider: ModelCheckpoint (save best model), EarlyStopping, ReduceLROnPlateau


In [ ]:
# (Optional) Define a custom learning rate schedule


In [ ]:
# Set training hyperparameters
num_epochs = None  # Choose an appropriate number


---

## Section 6: Training

Train your model using `model.fit()`. Your training should:

1. Train on the training set
2. Validate on the validation set after each epoch
3. Track training and validation loss and accuracy
4. Save the best model based on validation accuracy (via ModelCheckpoint callback)

### Best Practices:

- Use callbacks for checkpointing and early stopping
- The `history` object returned by `model.fit()` contains all training metrics
- Print or log progress so you can monitor training
- If training is slow, try reducing the number of epochs and iterating on architecture first

In [ ]:
# Train the model
# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=num_epochs,
#     callbacks=[...],
# )


In [ ]:
# If you used ModelCheckpoint, load the best model weights
# model.load_weights('best_model.weights.h5')
# Or if you saved the full model:
# model = tf.keras.models.load_model('best_model.keras')


In [ ]:
# Access training history for plotting
# history.history contains: 'loss', 'accuracy', 'val_loss', 'val_accuracy'


---

## Section 7: Training Visualization and Analysis

Visualize your training progress and analyze the results.

### Required Plots:

1. Training and validation loss over epochs
2. Training and validation accuracy over epochs

### Analysis Questions:

- Does the model show signs of overfitting? How can you tell?
- Did the learning rate schedule (if used) have a visible effect?
- At which epoch did the model achieve its best validation performance?

In [ ]:
# Plot training and validation loss
# If you used model.fit(), use: history.history['loss'], history.history['val_loss']


In [ ]:
# Plot training and validation accuracy
# If you used model.fit(), use: history.history['accuracy'], history.history['val_accuracy']


### Training Notes

Observations on your training curves—signs of overfitting/underfitting, best validation accuracy, effects of learning rate schedule, etc.

*Your notes:*



---

## Section 8: Model Evaluation

Load your best model and perform a thorough evaluation on the EMNIST test set.

### Required Analysis:

1. Report overall test accuracy
2. Generate a confusion matrix
3. Identify which letter pairs are most commonly confused
4. Visualize some correctly classified and misclassified examples

In [ ]:
# Load the best model (if not already in memory)


In [ ]:
# Evaluate on EMNIST test set
# model.evaluate(test_ds) or compute metrics manually with model.predict()


In [ ]:
# Generate predictions for confusion matrix


In [ ]:
# Plot confusion matrix
# Hint: Use sklearn.metrics.confusion_matrix and display with matplotlib or seaborn


In [ ]:
# Identify the most confused letter pairs
# Which letters does your model most frequently mix up?


In [ ]:
# Visualize correctly classified examples


In [ ]:
# Visualize misclassified examples
# Show the image, predicted label, and true label


### Evaluation Notes

Observations from your evaluation—test accuracy, which letters are confused, patterns in misclassifications, etc.

*Your notes:*



---

## Section 9: Experimentation Log

Document the experiments you ran while developing your model. Tracking your experiments helps you understand what works and why.

For each significant experiment, record:
- What you changed
- Your hypothesis for why it might help
- The parameter count
- The result (validation accuracy)
- What you learned

### Example Format:

| Experiment | Change | Hypothesis | Val Accuracy | Notes |
|------------|--------|------------|--------------|-------|
| Baseline | Initial CNN | - | 85.2% | Starting point |
| Exp 1 | Added dropout (0.5) | Reduce overfitting | 87.1% | Clear improvement |
| Exp 2 | Increased filters (32→64) | More capacity | 86.8% | No improvement, reverted |

### Your Experimentation Log

| Experiment | Change | Hypothesis | Val Accuracy | Notes |
|------------|--------|------------|--------------|-------|
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |

### Reflection

What worked? What surprised you? What would you try with more time?

*Your notes:*



---

## Section 10: Model Export and Submission

Run the cells below to export your model for grading. The export process wraps your model and its normalization into a single file so that all submissions are evaluated consistently.

### How it works:

1. You enter the normalization values you used during training (if any)
2. The submission cell wraps your model with a built-in `Rescaling` layer that applies normalization internally
3. The wrapped model is saved as a `.keras` file containing the full architecture and weights
4. Your grading script feeds raw [0, 1] pixel values; the exported model handles the rest

**This means you don't need to write a separate prediction script or worry about preprocessing mismatches.**

### Step 1: Enter your normalization values

If you normalized your training images beyond [0, 1] scaling (e.g., zero-centering with dataset mean/std), enter those values below. If you only divided by 255 with no further normalization, leave the defaults as `0.0` and `1.0`.

In [ ]:
# ============================================================
# FILL IN YOUR NORMALIZATION VALUES
# ============================================================
# If you normalized like: X_train = (X_train - 0.1722) / 0.3309
# Then set:    NORM_MEAN = 0.1722
#              NORM_STD  = 0.3309
#
# If you only scaled to [0, 1] (divided by 255) with no further
# normalization, leave these as-is.
NORM_MEAN = 0.0
NORM_STD = 1.0

### Step 2: Export your model

**Run this cell without modifications.** It wraps your model with the normalization values above, exports it as a single `.keras` file, and verifies the export is correct.

In [ ]:
#@title === SUBMISSION EXPORT (DO NOT MODIFY THIS CELL) ===

import tensorflow as tf
import numpy as np

# --- Parameter check ---
PARAMETER_LIMIT = 1_000_000
num_params = model.count_params()
print(f"Model parameters: {num_params:,}")
if num_params > PARAMETER_LIMIT:
    raise ValueError(
        f"❌ Model has {num_params:,} parameters, exceeding the "
        f"{PARAMETER_LIMIT:,} limit. Reduce your model size before submitting."
    )
print(f"✓ Parameter check passed ({num_params:,} ≤ {PARAMETER_LIMIT:,})")

# --- Build wrapped model with normalization baked in ---
# Uses Keras built-in Rescaling layer so no custom objects are needed on reload.
# Rescaling computes: output = input * scale + offset
# To normalize:  (x - mean) / std  =  x * (1/std) + (-mean/std)
use_norm = not (NORM_MEAN == 0.0 and NORM_STD == 1.0)

inputs = tf.keras.Input(shape=(28, 28, 1), name='image_input')
x = inputs
if use_norm:
    x = tf.keras.layers.Rescaling(
        scale=1.0 / NORM_STD,
        offset=-NORM_MEAN / NORM_STD,
        name='normalization'
    )(x)
outputs = model(x)
submission_model = tf.keras.Model(inputs=inputs, outputs=outputs, name='submission_model')

# --- Verify output shape ---
dummy = np.random.randn(1, 28, 28, 1).astype(np.float32)
expected_output = submission_model.predict(dummy, verbose=0)

assert expected_output.shape[1] == 26, (
    f"❌ Expected 26 output classes, got {expected_output.shape[1]}. "
    f"Make sure your output layer has 26 neurons (one per letter, 0-indexed A-Z)."
)

# --- Save ---
submission_model.save('submission_model.keras')

# --- Verify the saved model loads and matches ---
loaded = tf.keras.models.load_model('submission_model.keras')
reload_output = loaded.predict(dummy, verbose=0)

assert np.allclose(expected_output, reload_output, atol=1e-5), (
    "❌ Reload verification failed: outputs don't match. "
    "This shouldn't happen. Please reach out for help."
)

print(f"✓ Output shape: {reload_output.shape} (batch_size, 26)")
norm_str = f"Rescaling(mean={NORM_MEAN}, std={NORM_STD})" if use_norm else "None (raw [0,1] input)"
print(f"✓ Normalization baked in: {norm_str}")
print(f"✓ Model exported to: submission_model.keras")
print()
print("This file is ready to submit. It contains your full model,")
print("weights, and normalization, all in one file.")

### Step 3: Sanity check your exported model

This cell loads your exported model and tests it on sample images. It runs in two modes:

**Mode 1 (default): EMNIST test set.** Works immediately with no setup. Uses EMNIST test images with raw [0, 1] pixels (no normalization, no augmentation) to confirm the exported model produces reasonable predictions. This simulates what the grading script does.

**Mode 2: Sample images folder.** After uploading the sample images folder to Colab, set `USE_SAMPLE_IMAGES = True` and update `SAMPLE_FOLDER` to point to the uploaded folder. This tests your model on the same kind of independently-created handwritten images that will be used for final grading.

⚠️ **The sample images do NOT need the EMNIST orientation fix.** They are already correctly oriented. The preprocessing for sample images is just: convert to grayscale → scale to [0, 1]. The cell below handles this automatically.

In [ ]:
# ============================================================
# SANITY CHECK CONFIGURATION
# ============================================================
# Mode 1 (default): test on EMNIST data
# Mode 2: set USE_SAMPLE_IMAGES = True and point to your uploaded folder
USE_SAMPLE_IMAGES = False
SAMPLE_FOLDER = '/content/sample_images'  # Update this path if needed

In [ ]:
import os
import glob
from PIL import Image

loaded_model = tf.keras.models.load_model('submission_model.keras')

if USE_SAMPLE_IMAGES:
    # ----- Mode 2: Sample images from uploaded folder -----
    image_paths = sorted(glob.glob(os.path.join(SAMPLE_FOLDER, '*.png')))
    if not image_paths:
        image_paths = sorted(glob.glob(os.path.join(SAMPLE_FOLDER, '*.jpg')))
    assert len(image_paths) > 0, (
        f"No images found in {SAMPLE_FOLDER}. "
        f"Make sure you uploaded the folder and the path is correct."
    )

    images, true_letters, pred_letters = [], [], []
    for path in image_paths:
        # Extract ground-truth label from filename (first character)
        true_letter = os.path.basename(path)[0].upper()
        true_letters.append(true_letter)

        # Preprocessing: grayscale, resize to 28x28 (if needed), scale to [0,1]
        # NO EMNIST orientation fix — these images are already correctly oriented
        img = Image.open(path).convert('L')
        if img.size != (28, 28):
            img = img.resize((28, 28), Image.BILINEAR)
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_array = img_array.reshape(1, 28, 28, 1)  # (1, 28, 28, 1)
        images.append(img_array.squeeze())  # Store (28, 28) for display

        preds = loaded_model.predict(img_array, verbose=0)
        pred = preds.argmax(axis=1)[0]
        pred_letters.append(chr(pred + ord('A')))

    # Show results
    n_show = min(len(images), 16)
    n_cols = min(n_show, 8)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2.8 * n_rows))
    if n_rows == 1:
        axes = [axes] if n_cols == 1 else list(axes)
    else:
        axes = [ax for row in axes for ax in row]

    for i in range(n_show):
        ax = axes[i]
        ax.imshow(images[i], cmap='gray')
        color = 'green' if pred_letters[i] == true_letters[i] else 'red'
        ax.set_title(f"True: {true_letters[i]}\nPred: {pred_letters[i]}", color=color, fontsize=10)
        ax.axis('off')
    for i in range(n_show, len(axes)):
        axes[i].axis('off')

    correct = sum(p == t for p, t in zip(pred_letters, true_letters))
    title = f"Sample Images: {correct}/{len(true_letters)} correct ({100*correct/len(true_letters):.0f}%)"
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

    # Print per-image results
    print(f"\nResults on {len(true_letters)} sample images:")
    for i, (path, true, pred) in enumerate(zip(image_paths, true_letters, pred_letters)):
        status = "✓" if true == pred else "✗"
        print(f"  {status} {os.path.basename(path):20s}  true={true}  pred={pred}")
    print(f"\nAccuracy: {correct}/{len(true_letters)} ({100*correct/len(true_letters):.1f}%)")

else:
    # ----- Mode 1 (default): EMNIST test set -----
    # Use the raw test arrays from Section 2, scaled to [0,1]
    test_images_01 = test_images.astype(np.float32) / 255.0

    fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
    indices = np.random.choice(len(test_images_01), 8, replace=False)

    for ax, idx in zip(axes, indices):
        img = test_images_01[idx]  # (28, 28, 1)
        true_letter = chr((test_labels[idx] - 1) + ord('A'))

        preds = loaded_model.predict(img.reshape(1, 28, 28, 1), verbose=0)
        pred = preds.argmax(axis=1)[0]
        pred_letter = chr(pred + ord('A'))

        color = 'green' if pred_letter == true_letter else 'red'
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(f"True: {true_letter}\nPred: {pred_letter}", color=color, fontsize=10)
        ax.axis('off')

    plt.suptitle("Sanity Check: Exported Model on EMNIST Test Images", fontsize=12)
    plt.tight_layout()
    plt.show()

---

## Section 11: Final Summary

Provide a brief summary of your project.

### Project Summary

**Model Architecture:**
(Briefly describe your final architecture)


**Key Techniques Used:**
(List the main techniques: augmentation, regularization, etc.)


**Final Performance:**
- Training Accuracy:
- Validation Accuracy:
- Test Accuracy:
- Number of Parameters:

**Main Challenges and How You Addressed Them:**


**Key Takeaways:**


---

## Submission Checklist

Before submitting, verify:

- [ ] All code cells have been executed
- [ ] All analysis questions have been answered
- [ ] Normalization values in Step 1 match what you used during training
- [ ] `submission_model.keras` has been generated by the export cell
- [ ] Sanity check predictions look reasonable
- [ ] Experimentation log is filled out
- [ ] Final summary is complete

**Files to submit:**
1. This notebook (`.ipynb`)
2. `submission_model.keras`